# Budowanie baz wektorowych Terraria

Ten notebook tworzy dwie bazy Chroma:
- `chroma_wiki_db` z `wiki_data.jsonl`
- `chroma_structured_db` z `items.jsonl` i `recipes.jsonl`

`item_names.json` jest zapisywany osobno do fuzzy matchingu.

In [ ]:
# # Instalacja wymaganych bibliotek
# %pip install langchain langchain-core langchain-community langchain-ollama chromadb rapidfuzz


In [ ]:
import os
os.system("ollama pull nomic-embed-text")

In [ ]:
import json
import shutil
from pathlib import Path

from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings


In [ ]:
# Ścieżki do plików wejściowych
WIKI_FILE = Path("wiki_data.jsonl")
ITEMS_FILE = Path("items.jsonl")
RECIPES_FILE = Path("recipes.jsonl")

# Foldery wynikowych baz Chroma
WIKI_DB_DIR = Path("chroma_wiki_db")
STRUCTURED_DB_DIR = Path("chroma_structured_db")

# Lista nazw itemów do fuzzy matchingu
ITEM_NAMES_FILE = Path("item_names.json")


In [ ]:
def read_jsonl(path: Path):
    # Wczytuje plik JSONL, czyli jeden JSON w jednej linii
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            records.append(json.loads(line))

    return records


def safe_text(value):
    # Zamienia listy, słowniki i puste wartości na tekst
    if value is None:
        return ""

    if isinstance(value, list):
        return ", ".join(str(x) for x in value)

    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)

    return str(value)


def clean_metadata(metadata: dict):
    # Chroma przyjmuje w metadata tylko proste typy danych
    cleaned = {}

    for key, value in metadata.items():
        if value is None:
            cleaned[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            cleaned[key] = value
        else:
            cleaned[key] = safe_text(value)

    return cleaned


In [ ]:
def split_text(text: str, chunk_size=1000, chunk_overlap=150):
    # Dzieli długi tekst na mniejsze fragmenty z zachodzeniem na siebie
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - chunk_overlap

    return chunks


def split_documents_simple(documents, chunk_size=1000, chunk_overlap=150):
    # Dzieli dokumenty wiki na chunki, żeby wyszukiwanie trafiało w konkretne fragmenty
    split_docs = []

    for doc in documents:
        chunks = split_text(doc.page_content, chunk_size, chunk_overlap)

        for index, chunk in enumerate(chunks):
            metadata = dict(doc.metadata)
            metadata["chunk"] = index

            split_docs.append(Document(
                page_content=chunk,
                metadata=clean_metadata(metadata)
            ))

    return split_docs


In [ ]:
def load_wiki_documents(path: Path):
    # Wczytuje teksty stron wiki jako dokumenty do bazy opisowej
    records = read_jsonl(path)
    documents = []

    for record in records:
        content = (
            record.get("content")
            or record.get("text")
            or record.get("body")
            or ""
        )

        title = (
            record.get("title")
            or record.get("page")
            or record.get("name")
            or "unknown"
        )

        if not content:
            continue

        documents.append(Document(
            page_content=f"Page: {title}{content}",
            metadata=clean_metadata({
                "source": "wiki",
                "title": title,
                "url": record.get("url", ""),
                "page": record.get("page", title),
            })
        ))

    return documents


In [ ]:
def load_items_documents(path: Path):
    # Wczytuje dane itemów, czyli statystyki, typy, opisy i wartości
    records = read_jsonl(path)
    documents = []

    useful_fields = [
        "name", "itemid", "internalname", "type", "tag",
        "damage", "damagetype", "defense", "velocity",
        "knockback", "rare", "buy", "sell",
        "axe", "pick", "hammer", "fishing", "bait",
        "bonus", "toolspeed", "usetime", "critical",
        "tooltip", "mana", "hheal", "mheal",
        "buffs", "debuffs", "hardmode", "consumable",
        "placeable"
    ]

    for item in records:
        name = item.get("name")

        if not name:
            continue

        lines = [f"Item stats for: {name}"]

        for field in useful_fields:
            value = item.get(field)

            if value not in [None, "", [], {}]:
                lines.append(f"{field}: {safe_text(value)}")

        documents.append(Document(
            page_content="".join(lines),
            metadata=clean_metadata({
                "source": "structured",
                "type": "item_stats",
                "item_name": name,
                "itemid": item.get("itemid", ""),
                "page": item.get("page", ""),
            })
        ))

    return documents


In [ ]:
def load_recipes_documents(path: Path):
    # Wczytuje receptury craftingu jako osobne dokumenty
    records = read_jsonl(path)
    documents = []

    for recipe in records:
        result = recipe.get("result")

        if not result:
            continue

        ingredients = parse_ingredients_from_recipe(recipe)
        ingredients_text = ", ".join(
            f"{amount}x {name}"
            for name, amount in ingredients
        )
        stations = recipe.get("station_list", [])
        legacy = str(recipe.get("legacy", ""))

        text = "\n".join([
    f"Recipe for: {result}",
    f"Result amount: {safe_text(recipe.get('amount', 1))}",
    f"Ingredients: {ingredients_text}",
    f"Crafting station: {safe_text(stations)}",
    f"Legacy recipe: {legacy}",
    f"Version: {safe_text(recipe.get('version', ''))}",
])

        documents.append(Document(
            page_content=text,
            metadata=clean_metadata({
                "source": "structured",
                "type": "recipe",
                "item_name": result,
                "resultid": recipe.get("resultid", ""),
                "legacy": legacy,
                "page": recipe.get("page", ""),
            })
        ))

    return documents


In [ ]:
import re

def parse_ingredient(raw: str):
    # Rozdziela np. "Iron Bar8" na ("Iron Bar", 8)
    raw = str(raw).strip()

    match = re.match(r"^(.*?)(\d+)$", raw)

    if match:
        name = match.group(1).strip()
        amount = int(match.group(2))
        return name, amount

    return raw, 1


def parse_ingredients_from_recipe(recipe):
    # Bierze składniki z pola ings/args, bo tam jest ilość składników
    raw_ingredients = (
        recipe.get("ings")
        or recipe.get("args")
        or recipe.get("ingredients")
        or recipe.get("ingredients_list")
        or []
    )

    if isinstance(raw_ingredients, str):
        parts = re.split(r"\s*,\s*|\s*;\s*|\s*\|\s*", raw_ingredients)
    else:
        parts = raw_ingredients

    ingredients = []

    for part in parts:
        if not part:
            continue

        name, amount = parse_ingredient(part)
        ingredients.append((name, amount))

    return ingredients

In [ ]:
def save_item_names(items_documents, recipes_documents, output_path: Path):
    # Zapisuje wszystkie nazwy itemów do późniejszego fuzzy matchingu
    item_names = set()

    for doc in items_documents + recipes_documents:
        name = doc.metadata.get("item_name")

        if name:
            item_names.add(name)

    item_names = sorted(item_names)

    with output_path.open("w", encoding="utf-8") as file:
        json.dump(item_names, file, ensure_ascii=False, indent=2)

    return item_names


def reset_database_folder(path: Path):
    # Usuwa starą bazę, żeby nowe dane nie mieszały się ze starymi
    if path.exists():
        print(f"Usuwam starą bazę: {path}")
        shutil.rmtree(path)


def build_database(documents, persist_directory: Path, embeddings, batch_size=25):
    # Tworzy bazę partiami, żeby kernel nie padał przy dużej liczbie dokumentów
    db = Chroma(
        persist_directory=str(persist_directory),
        embedding_function=embeddings
    )

    for start in range(0, len(documents), batch_size):
        end = min(start + batch_size, len(documents))
        batch = documents[start:end]

        print(f"Dodaję dokumenty {start}-{end} / {len(documents)}")
        db.add_documents(batch)

    if hasattr(db, "persist"):
        db.persist()

    return db


In [ ]:
print("Wczytuję dane wiki...")
wiki_documents = load_wiki_documents(WIKI_FILE)

print("Wczytuję dane itemów...")
items_documents = load_items_documents(ITEMS_FILE)

print("Wczytuję receptury...")
recipes_documents = load_recipes_documents(RECIPES_FILE)

structured_documents = items_documents + recipes_documents

print("Zapisuję listę nazw itemów...")
item_names = save_item_names(items_documents, recipes_documents, ITEM_NAMES_FILE)

print("Dzielę teksty wiki na fragmenty...")
wiki_pieces = split_documents_simple(
    wiki_documents,
    chunk_size=1000,
    chunk_overlap=150
)

structured_pieces = structured_documents

print("Wiki documents:", len(wiki_documents))
print("Wiki chunks:", len(wiki_pieces))
print("Item documents:", len(items_documents))
print("Recipe documents:", len(recipes_documents))
print("Structured documents:", len(structured_pieces))
print("Item names:", len(item_names))


In [ ]:
print("Ładuję embeddingi z Ollama...")
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


In [ ]:
#reset_database_folder(WIKI_DB_DIR)
reset_database_folder(STRUCTURED_DB_DIR)

# print("Buduję bazę wiki...")
# wiki_db = build_database(
#     documents=wiki_pieces,
#     persist_directory=WIKI_DB_DIR,
#     embeddings=embeddings
# )

print("Buduję bazę structured...")
structured_db = build_database(
    documents=structured_pieces,
    persist_directory=STRUCTURED_DB_DIR,
    embeddings=embeddings
)

print("Gotowe.")
print("Utworzono:")
print(f"- {WIKI_DB_DIR}")
print(f"- {STRUCTURED_DB_DIR}")
print(f"- {ITEM_NAMES_FILE}")
